# **Similitud Semántica (STS)**  
El objetivo de esta práctica es ajustar un modelo Transformer (distilroberta-base) para predecir la simlitud semántica entre pares de frases del dataset stsbenckmark. Se utilizará un enfoque Bi-Encoder. Posteriormente se evaluará mediante la correlación de Pearson.

### **1. Carga y exploración del dataset**  
El dataset stsbenchmark contiene pares de frases y una etiqueta numérica que determina su similitud, junto a otros datos que no resultarán tan relevantes para el objetivo de esta práctica.


In [ ]:
from datasets import load_dataset
import pandas as pd

# Cargamos el dataset
dataset = load_dataset("mteb/stsbenchmark-sts")
print(dataset['train'][0])
print(f"\nColumnas: {dataset['train'].column_names}")

### **2. Selección y configuración del modelo**  
Se utilizará un enfoque Bi-Encoder mediante la librería sentence-transformers. Este modelo procesará cada frase por separado por generar un vector (embedding) y luego calcula la similtud entre ellos.


In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

# Definimos el modelo base
model = SentenceTransformer('distilroberta-base')

train_ejemplos = []

for row in dataset['train']:
  # Normalizamos el score para poder usarlo con SentenceTransformers que espera valores [0-1]
  score = float(row['score'])/5.0
  train_ejemplos.append(InputExample(texts=[row['sentence1'], row['sentence2']], label=score))

# Creamos el DataLoader
train_dataloader = DataLoader(train_ejemplos, shuffle=True, batch_size=16)

#Definimos la funcón de pérdida
train_loss = losses.CosineSimilarityLoss(model)


Utilizamos la librería sentence-transformers para realizar el fine-tuning. La función de pérdida seleccionada es CosineSimilarityLoss, ya que nuestra tarea es de regresión.

### **3. Entrenamiento**  

In [ ]:
sentence1 = [row['sentence1'] for row in dataset['test']]
sentence2 = [row['sentence2'] for row in dataset['test']]
label = [float(row['score']) for row in dataset['test']]

In [ ]:
from scipy.stats import pearsonr
from sklearn.metrics.pairwise import paired_cosine_distances

print("Iniciando Experimento 1 (1 época)...")
model.fit(train_objectives=[(train_dataloader, train_loss)],
          epochs=1,
          warmup_steps=100)

# Evaluación para el Experimento 1
emb1_exp1 = model.encode(sentence1)
emb2_exp1 = model.encode(sentence2)
scores_exp1 = 1 - paired_cosine_distances(emb1_exp1, emb2_exp1)
pearson_exp1, _ = pearsonr(scores_exp1 * 5, label)

print(f"Correlación de Pearson (1 época): {pearson_exp1:.4f}")

print("\nContinuando entrenamiento para el Experimento 2 (hasta 4 épocas)...")
model.fit(train_objectives=[(train_dataloader, train_loss)],
          epochs=3, # Sumamos 3 a la que ya teníamos
          warmup_steps=100)

# Evaluación final para el Experimento 2
embedding1 = model.encode(sentence1)
embedding2 = model.encode(sentence2)
cosine_scores = 1 - paired_cosine_distances(embedding1, embedding2)
pearson_corr, _ = pearsonr(cosine_scores * 5, label)

print(f"Correlación de Pearson Final (4 épocas): {pearson_corr:.4f}")

### **4. Evaluación y métrica de Pearson**  
A continuación se mostrará la eficacia del modelo mediante la Correlación de Pearson sobre el conjunto de test.

In [ ]:
# Obtenemos los embeddings (vectores) y calculamos la similitud del coseno
embedding1 = model.encode(sentence1)
embedding2 = model.encode(sentence2)

# La distancia del coseno es 1-similitud, por eso:
cosine_scores = 1 - paired_cosine_distances(embedding1, embedding2)

# Caluclamos la correlación de Pearson comparando las predicciones con las reales
pearson_corr, _ = pearsonr(cosine_scores * 5, label)

print(f"Resultados Finales:")
print(f"Correlación de Pearson en Test: {pearson_corr:.4f}")

A continuación, visualizamos la relación entre las predicciones del modelo final (4 épocas) y las etiquetas reales. Una mayor concentración de puntos en la diagonal indica una mayor eficacia del modelo.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración del estilo
plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")

# Crear el gráfico de dispersión con línea de regresión
# x: tus predicciones (escaladas a 0-5), y: las etiquetas reales del dataset
sns.regplot(x=cosine_scores * 5, y=label,
            scatter_kws={'alpha':0.3, 'color': 'blue'},
            line_kws={'color':'red'})

plt.title(f'Visualización de la Correlación: {pearson_corr:.4f}', fontsize=15)
plt.xlabel('Predicciones del Modelo (Similitud del Coseno x 5)', fontsize=12)
plt.ylabel('Etiquetas Reales (STSb)', fontsize=12)
plt.xlim(0, 5)
plt.ylim(0, 5)

plt.show()

### **5. Demostración Final**  
A continuación se muestran 6 ejemplos inventados. El objetivo es verificar si las predicciones del modelo se alinean con la similitud semántica real de las frases

In [ ]:
# Lista de ejemplos optimizada para mostrar la progresión de 0 a 5
ejemplos_escala = [
    ("A man is playing the piano.", "A man plays the piano."),                  # Nivel 5: Identidad total
    ("A man is eating food.", "A man is eating a meal."),                       # Nivel 4: Muy similares
    ("A man is cutting an apple.", "A man is slicing a vegetable."),            # Nivel 3: Acción similar, objeto relacionado
    ("A man is walking through the woods.", "A man is walking on the beach."),  # Nivel 2: Misma acción, distinto lugar
    ("A man is cooking.", "A man is sleeping."),                                # Nivel 1: Mismo sujeto, distinta acción
    ("The sky is blue.", "The stock market crashed.")                           # Nivel 0: Sin relación")
]

print("--- Demostración: Predicciones por niveles (0-5) ---\n")

from sklearn.metrics.pairwise import cosine_similarity

for s1, s2 in ejemplos_escala:
    emb1 = model.encode([s1])
    emb2 = model.encode([s2])
    sim = cosine_similarity(emb1, emb2)[0][0]
    score_final = max(0, min(5, sim * 5))

    print(f"Frase 1: {s1}")
    print(f"Frase 2: {s2}")
    print(f"Predicción: {score_final:.2f}/5")
    print("-" * 50)

### **6. Tabla Comparativa**  


In [ ]:
import pandas as pd

# Tabla comparativa exigida por el enunciado
data_resultados = {
    "Configuración": ["DistilRoBERTa (Baseline)", "DistilRoBERTa (Fine-tuned)"],
    "Épocas Totales": [1, 4],
    "Pearson Correlation": [f"{pearson_exp1:.4f}", f"{pearson_corr:.4f}"]
}

tabla_final = pd.DataFrame(data_resultados)
display(tabla_final)

### **7. Conclusiones**  


1.   **Mejora de aprendizaje:** se observa que al aumentar las épocas de 1 a 4, la correlación de Pearson mejora, lo que indica que el modelo ajusta mejoir los embeddings en el espacio vectorial.
2.   **Arquitectura:** el uso de Bi-Encoder es eficiente para esta tarea, permitiendo comparar semánticamente frases mediante la similitud del coseno de forma rápida.
3. **Validación:** la demostración final muestra que el modelo asigna puntuaciones coherentes con la percepción humana, incluso en frases no vistas durante el entrenamiento



